# F1-P1 private 25-record development smoke

This private Kaggle workflow completes the final technical gate for the already trained F1-P1 natural-data pilot. It reloads only the frozen selected checkpoint and generates on exactly 25 frozen QALB-2014 L1 development records. It does not train, select a checkpoint, tune a prompt, load QALB test, or load Nahw-Passage. Raw responses remain private; only a text-free audit summary may be committed.

## 1. Runtime and reproducible dependencies
Use a private Kaggle notebook with a free NVIDIA GPU and Internet enabled. The prior private full-training kernel output must be attached as a notebook source.

In [ ]:
import os, subprocess, sys
assigned_gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip()
import PIL
P100_STACK = {'torch': '2.6.0', 'torchvision': '0.21.0', 'xformers': '0.0.29.post3', 'torchao': '0.16.0', 'numpy': '2.0.2', 'pillow': PIL.__version__, 'index': 'https://download.pytorch.org/whl/cu124'}
if 'P100' in assigned_gpu_name:
    os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall', f"torch=={P100_STACK['torch']}", f"torchvision=={P100_STACK['torchvision']}", f"xformers=={P100_STACK['xformers']}", f"numpy=={P100_STACK['numpy']}", f"Pillow=={P100_STACK['pillow']}", '--index-url', P100_STACK['index']], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', f"torchao=={P100_STACK['torchao']}"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU assigned. Enable a Kaggle GPU accelerator and restart.')
gpu = torch.cuda.get_device_properties(0)
torch.ones(1, device='cuda').sum().item(); torch.cuda.synchronize()
print({'gpu': gpu.name, 'cuda_capability': f'{gpu.major}.{gpu.minor}', 'total_vram_bytes': gpu.total_memory})

## 2. Safe repository setup
Clone the public repository into Kaggle working storage. Private data and checkpoint files remain read-only notebook inputs.

In [ ]:
from pathlib import Path
RUNTIME_ROOT = Path('/kaggle/working')
REPO = RUNTIME_ROOT / 'Musahhih'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ALBA7OOTH-Research-Lab/Musahhih.git', str(REPO)], check=True)
else:
    dirty = subprocess.run(['git', '-C', str(REPO), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('Existing repository has local changes; refusing to overwrite them.')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
GIT_SHA = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print({'repository': str(REPO), 'git_sha': GIT_SHA})

In [ ]:
import re
torch_minor = re.match(r'[0-9]+\.[0-9]+', torch.__version__).group(0)
xformers_version = {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2','2.6':'0.0.29.post3'}.get(torch_minor, '0.0.34')
def pip_install(*items):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *items], check=True)
pip_install('sentencepiece', 'protobuf', 'datasets==4.3.0', 'huggingface_hub>=0.34.0', 'hf_transfer')
pip_install('--no-deps', 'unsloth_zoo', 'bitsandbytes', 'accelerate', f'xformers=={xformers_version}', 'peft', 'trl', 'triton', 'unsloth')
pip_install('--no-deps', '--upgrade', 'torchao==0.16.0' if torch_minor == '2.6' else 'torchao>=0.16.0')
pip_install('transformers==4.56.2')
pip_install('--no-deps', 'trl==0.22.2')
import importlib.metadata as md, json, platform
PACKAGE_NAMES = ['torch','transformers','unsloth','accelerate','peft','trl','datasets','bitsandbytes']
VERSIONS = {name: md.version(name) for name in PACKAGE_NAMES}
RUNTIME = {'python': platform.python_version(), 'cuda': torch.version.cuda, 'gpu': gpu.name, 'cuda_capability': f'{gpu.major}.{gpu.minor}', 'packages': VERSIONS}
print(json.dumps(RUNTIME, indent=2))

## 3. Locate and validate the frozen private artifacts
This cell discovers only the expected filenames inside attached private Kaggle inputs. It prints hashes and counts, never corpus text.

In [ ]:
import hashlib
from scripts.f1_training_utils import MODEL_REVISION, MAX_SEQUENCE_LENGTH, sha256_file, validate_private_jsonl
EXPERIMENT_ID = 'F1-P1__gemma3-4b-it__qalb14-dev__s3407__r01'
PRIVATE_INPUT_ROOT = Path('/kaggle/input')
selection_candidates = [path for path in PRIVATE_INPUT_ROOT.rglob('checkpoint_selection.json') if path.parent.name == EXPERIMENT_ID]
if len(selection_candidates) != 1:
    raise RuntimeError(f'Expected one private F1-P1 checkpoint selection artifact; observed {len(selection_candidates)}')
CHECKPOINT_SELECTION_PATH = selection_candidates[0]
PRIVATE_RUN_DIR = CHECKPOINT_SELECTION_PATH.parent
CHECKPOINT_SELECTION = json.loads(CHECKPOINT_SELECTION_PATH.read_text(encoding='utf-8'))
if CHECKPOINT_SELECTION.get('selected_checkpoint') != 'checkpoint-250' or len(CHECKPOINT_SELECTION.get('evaluations', [])) != 2:
    raise RuntimeError('Private checkpoint-selection artifact does not match the frozen completed run.')
SELECTED_CHECKPOINT = PRIVATE_RUN_DIR / CHECKPOINT_SELECTION['selected_checkpoint']
if not SELECTED_CHECKPOINT.is_dir():
    raise RuntimeError('Selected private checkpoint directory is missing.')
dev_candidates = [path for path in PRIVATE_INPUT_ROOT.rglob('dev_records.jsonl') if path.parent.name == 'f1_p1']
if len(dev_candidates) != 1:
    raise RuntimeError(f'Expected one private F1-P1 development record file; observed {len(dev_candidates)}')
DEV_PATH = dev_candidates[0]
DEV_INPUT = validate_private_jsonl(DEV_PATH, 'development', 975)
PUBLIC_TRAINING_SUMMARY = json.loads(Path('results/f1_p1_full_training_summary.json').read_text(encoding='utf-8'))
with DEV_PATH.open('r', encoding='utf-8') as stream:
    private_dev = [json.loads(line) for line in stream]
selection_bytes = ''.join(json.dumps({'record_id': row['record_id']}, ensure_ascii=False, sort_keys=True) + '\n' for row in private_dev).encode('utf-8')
development_selection_sha256 = hashlib.sha256(selection_bytes).hexdigest()
if development_selection_sha256 != PUBLIC_TRAINING_SUMMARY['data']['development_selection_sha256']:
    raise RuntimeError('Private development selection hash does not match the completed training summary.')
print({'checkpoint': SELECTED_CHECKPOINT.name, 'development': DEV_INPUT, 'selection_hash_matches': True, 'contains_corpus_text': False})

## 4. Reload the selected adapter
Load the immutable selected checkpoint in 4-bit mode and switch to inference. No LoRA adapter is created and no optimizer or trainer is constructed.

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
model, processor = FastModel.from_pretrained(model_name=str(SELECTED_CHECKPOINT), max_seq_length=MAX_SEQUENCE_LENGTH, load_in_4bit=True)
processor = get_chat_template(processor, chat_template='gemma-3')
FastModel.for_inference(model)
print({'selected_checkpoint': SELECTED_CHECKPOINT.name, 'model_revision': MODEL_REVISION, 'load_in_4bit': True})

## 5. Deliberate frozen 25-record generation gate
The committed notebook is disabled by default. For the authorized private run, deliberately enable the exact confirmation string. Generation is greedy with `do_sample=False`, no temperature argument, and `max_new_tokens=256`. Raw responses are written privately and never printed.

In [ ]:
RUN_PRIVATE_DEV_SMOKE = False
PRIVATE_DEV_SMOKE_CONFIRMATION = ''
REQUIRED_CONFIRMATION = 'RUN_F1_P1_PRIVATE_DEV_SMOKE_25'
if RUN_PRIVATE_DEV_SMOKE:
    if PRIVATE_DEV_SMOKE_CONFIRMATION != REQUIRED_CONFIRMATION:
        raise RuntimeError('Exact private development smoke confirmation is missing.')
    from collections import Counter
    from scripts.nahw_baseline_utils import parse_model_response
    order = sorted(range(len(private_dev)), key=lambda index: hashlib.sha256(f"F1-P1-dev-smoke|3407|{private_dev[index]['record_id']}".encode('utf-8')).hexdigest())[:25]
    selected_id_bytes = ''.join(private_dev[index]['record_id'] + '\n' for index in order).encode('utf-8')
    selected_record_ids_sha256 = hashlib.sha256(selected_id_bytes).hexdigest()
    OUTPUT_DIR = Path('/kaggle/working/f1_p1_private_dev_smoke')
    if OUTPUT_DIR.exists():
        raise RuntimeError('Private smoke output directory already exists; refusing to overwrite it.')
    OUTPUT_DIR.mkdir(parents=True)
    predictions_path = OUTPUT_DIR / 'dev_predictions.jsonl'
    warning_counts = Counter(); exact_matches = 0; empty_outputs = 0
    with predictions_path.open('w', encoding='utf-8') as output_stream:
        for position, index in enumerate(order, 1):
            row = private_dev[index]
            inference_messages = [{'role': message['role'], 'content': [{'type': 'text', 'text': message['content']}]} for message in row['prompt']]
            inputs = processor.apply_chat_template(inference_messages, tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=True).to('cuda')
            with torch.inference_mode():
                generated = model.generate(**inputs, do_sample=False, max_new_tokens=256)
            response = processor.decode(generated[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
            parsed, warnings = parse_model_response(response)
            gold = row['completion'][0]['content']
            exact_match = parsed == gold
            exact_matches += int(exact_match); empty_outputs += int(not parsed); warning_counts.update(warnings)
            private_result = {'record_id': row['record_id'], 'source_sha256': hashlib.sha256(row['prompt'][0]['content'].encode('utf-8')).hexdigest(), 'gold_sha256': hashlib.sha256(gold.encode('utf-8')).hexdigest(), 'raw_response': response, 'parsed_response': parsed, 'parsing_warnings': warnings, 'exact_match': exact_match}
            output_stream.write(json.dumps(private_result, ensure_ascii=False, sort_keys=True) + '\n')
            print({'completed': position, 'of': 25})
    predictions_sha256 = sha256_file(predictions_path)
    summary = {'protocol_id': 'F1-P1', 'gate': 'private_25_record_development_smoke', 'completed': True, 'contains_corpus_text': False, 'selected_checkpoint': SELECTED_CHECKPOINT.name, 'model_revision': MODEL_REVISION, 'records_expected': 25, 'records_completed': 25, 'selected_record_ids_sha256': selected_record_ids_sha256, 'predictions_sha256': predictions_sha256, 'checkpoint_selection_sha256': sha256_file(CHECKPOINT_SELECTION_PATH), 'adapter_config_sha256': sha256_file(SELECTED_CHECKPOINT / 'adapter_config.json'), 'decoding': {'do_sample': False, 'temperature_argument': None, 'max_new_tokens': 256}, 'parser_warning_counts': dict(sorted(warning_counts.items())), 'empty_output_count': empty_outputs, 'private_exact_match_count': exact_matches, 'private_exact_match_metric_publication_allowed': False, 'runtime': RUNTIME, 'git_sha': GIT_SHA, 'qalb_test_used': False, 'nahw_passage_used': False, 'checkpoint_selection_changed': False}
    summary_path = OUTPUT_DIR / 'f1_p1_dev_smoke_summary.json'
    summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    print(json.dumps({key: value for key, value in summary.items() if key != 'private_exact_match_count'}, indent=2))
else:
    print('Private development smoke is disabled; no generation was executed.')

## 6. Private export and public audit
Keep `dev_predictions.jsonl`, the selected adapter, and full logs private. Download only the text-free summary for repository audit. Do not publish the private exact-match count; this smoke is a pipeline gate, not a checkpoint-selection or paper result.